<a href="https://colab.research.google.com/github/abosameh/colab/blob/main/Cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Verify GPU acceleration is active

!sudo apt-get install -y pciutils zstd
# 2. Install the official Linux Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Download the official Cloudflare Tunnel Linux client
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n Setup finished! Head to Cell 2.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 53 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 kB in 0s (8,697 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/D

In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [4]:
import subprocess
import time
import re
import os

# Set environment variable so Ollama binds correctly to all external traffic loops
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

print("🔄 Launching Ollama Server...")
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)  # Allow system initialization

print("🌐 Spawning Cloudflare Quick Tunnel...")
# Start cloudflared to forward traffic and correctly inject host headers
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434', '--http-host-header', 'localhost:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)
tunnel_url = None

# Scan cloudflared's terminal logs to find your generated public domain URL
for _ in range(50):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print(f"\n🚀 SUCCESS! Your local PC can connect to this endpoint:")
    print(f"👉 {tunnel_url} 👈")
    print("\nKeep this notebook running. Copy the URL above and pass it to your local scripts!")
else:
    print("\n❌ Failed to capture a tunnel URL automatically. Run tunnel manually or check logs.")

🔄 Launching Ollama Server...
🌐 Spawning Cloudflare Quick Tunnel...

🚀 SUCCESS! Your local PC can connect to this endpoint:
👉 https://majority-excellence-merit-consideration.trycloudflare.com 👈

Keep this notebook running. Copy the URL above and pass it to your local scripts!


In [5]:
# Pull your models here. You can swap 'llama3.2' for 'deepseek-r1:8b' or 'gemma2'
#!ollama rm hf.co/mradermacher/Mythos-nano-GGUF:Q4_K_M
!ollama pull nutboy02/Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-uncenfull

In [6]:
!ollama list

NAME                                                                     ID              SIZE     MODIFIED      
nutboy02/Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-uncenfull:latest    48f4ff9730f2    28 GB    5 seconds ago    


In [ ]:
import requests
import json
import time

# Use your tunnel URL here
tunnel_url = "https://majority-excellence-merit-consideration.trycloudflare.com"

# Give Ollama some time to fully load the large model
time.sleep(60)

response = requests.post(f"{tunnel_url}/api/generate",
                        json={
                            "model": "nutboy02/Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-uncenfull:latest",
                            "prompt": "Hello, how are you?",
                            "stream": False
                        })

print(response.json()["response"])

In [ ]:
# export OLLAMA_HOST="https://followed-august-clara-tap.trycloudflare.com"
#export ANTHROPIC_AUTH_TOKEN=ollama
#export ANTHROPIC_API_KEY=""
#export ANTHROPIC_BASE_URL=https://majority-excellence-merit-consideration.trycloudflare.com